# improved_v2: trích xuất khái niệm y khoa từ bệnh án tiếng Việt

GLiNER với ngưỡng theo từng type, selector hai teacher Qwen (sửa type và bổ sung span), linking exact-alias (chỉ khớp tuyệt đối). Đây là bản đạt điểm cao nhất: **27.8786**.

Notebook chạy trọn trên một **Colab T4 (16 GB)**: hai teacher nạp ở **4-bit**, compute dtype đặt `float16` vì T4 không hỗ trợ bfloat16 hiệu quả. Kiến trúc chi tiết ở `docs/02_method.md`.

## 1. Kiểm tra runtime

Runtime, Change runtime type, chọn **T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Clone và cài đặt

Colab đã có torch bản CUDA. `pyproject.toml` là nguồn phụ thuộc duy nhất. Cài thêm `.[quant]` để nạp hai teacher ở 4-bit.

In [ ]:
!git clone https://github.com/AIVIETNAM-AIO-DinhBao/ViClinicalIE_2 medextract
%cd medextract
!pip install -e ".[quant]"    # bitsandbytes cho chế độ 4-bit

## 3. Self-check

Không cần GPU, không cần knowledge base. Phải in PASS cho cả bốn mục CONFIG / IMPORTS / SCHEMA / PATHS.

In [ ]:
!python scripts/selfcheck.py

## 4. Knowledge base cho bước linking

`improved_v2` chỉ dùng exact-alias lookup trên hai bảng parquet, **không** cần SapBERT và **không** cần FAISS index, nên chỉ cần hai lệnh build dưới đây. Danh mục ICD-10 tiếng Việt (TT06) **đã đi kèm repo** tại `data/kb/raw/`, nên bạn chỉ cần tải RxNorm và đặt vào `data/kb/raw/RXNCONSO.RRF` (xem `INSTALL.md` cho các nguồn RxNorm).

In [ ]:
from pathlib import Path

raw_dir = Path("data/kb/raw")
icd_file = raw_dir / "Phu_luc_Bang_danh_muc_ICD10_FINAL_TT06_2026.xlsx"
rxnorm_file = raw_dir / "RXNCONSO.RRF"
print("KB raw files:")
!ls -lh data/kb/raw
if not icd_file.exists():
    raise FileNotFoundError(f"Missing ICD-10 TT06 file from repo: {icd_file}")
if not rxnorm_file.exists():
    raise FileNotFoundError(
        "Missing RXNCONSO.RRF. This file is license-gated, so it is not committed to the public repo. "
        "Upload it to data/kb/raw/RXNCONSO.RRF or copy it from Drive before running build_rxnorm."
    )
print("OK: required KB raw files are available.")


In [ ]:
!python -m medextract.kb.build_icd    --tt06    # -> data/kb/processed/icd_terms_v2.parquet
!python -m medextract.kb.build_rxnorm --v2      # -> data/kb/processed/rxnorm_terms_v2.parquet

## 5. Config phụ cho Colab T4

Hai teacher nạp 4-bit, compute dtype `float16`. Kế thừa toàn bộ `configs/improved_v2.yaml`, chỉ ghi đè đường dẫn model và quantization.

In [ ]:
import pathlib

# Repo id của hai teacher trên Hugging Face Hub.
# Nếu bạn đã tải sẵn trọng số về máy, thay bằng đường dẫn cục bộ.
PRIMARY_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
SECONDARY_MODEL = "Qwen/Qwen3.5-4B"

override = f"""# Config phụ cho Colab T4: hai teacher nạp 4-bit, compute dtype float16.
# Kế thừa toàn bộ improved_v2, chỉ ghi đè đường dẫn model và quantization.
extends: configs/improved_v2.yaml
consensus_selector:
  primary_model: {PRIMARY_MODEL}
  secondary_model: {SECONDARY_MODEL}
  primary_device: cuda:0
  secondary_device: cuda:0
  batch_size: 16
quantization:
  mode: 4bit
  compute_dtype: float16
  double_quant: true
"""

pathlib.Path("colab_t4.yaml").write_text(override, encoding="utf-8")
print(override)

## 6. Chạy đầy đủ và đóng gói bản nộp

Hai bệnh án mẫu đã có sẵn trong `examples/input/`; cell dưới copy chúng vào `data/input/` để chạy. `--zip` ghi `out/improved_v2/submission.zip`, các file JSON nằm phẳng, không có thư mục con.

In [ ]:
from pathlib import Path

input_dir = Path("data/input")
txt_files = sorted(input_dir.glob("*.txt"), key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem)
print(f"Found {len(txt_files)} input .txt files in {input_dir}")
print([p.name for p in txt_files[:10]])
if len(txt_files) != 100:
    raise RuntimeError(f"Expected 100 input files in {input_dir}, found {len(txt_files)}")

!rm -rf out/improved_v2
!python run.py --config colab_t4.yaml --input data/input \
               --output out/improved_v2 --zip

!echo "JSON outputs:"
!find out/improved_v2 -maxdepth 1 -type f -name "*.json" | wc -l
!ls -lh out/improved_v2/submission.zip


## 7. Xem một mẫu output

In [ ]:
import json, pathlib

p = pathlib.Path("out/improved_v2/001.json")
data = json.load(open(p, encoding="utf-8"))
print(f"{p.name}: {len(data)} concept(s)\n")
print(json.dumps(data[:3], ensure_ascii=False, indent=2))

## 8. Chấm điểm local

`score.py` là bản đọc lại công thức của Ban Tổ chức để xếp hạng hai lần chạy local, không phải bộ chấm chính thức. Chuẩn bị thư mục nhãn dạng `<thư mục nhãn>/{stem}.json` cùng schema với bản nộp, rồi:

```bash
python score.py --pred out/improved_v2 --gold <thư mục nhãn> -v
```